In [2]:
import pandas as pd
import numpy as np
import msgspec
from pathlib import Path
from pprint import pprint

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()

def read_json(file_path, jsonl=False):
    with open(file_path, 'r') as file:
        data = file.read()
    if jsonl:
        output = decoder.decode_lines(data)
    else:
        output = decoder.decode(data)
        
    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output
    

# Turn nobel prize json files from TS-Retriever to Tevatron JSONL files

In [ ]:
"""
2.Exactly Matched Dataset

If the relevancy of a passage is judged by answer exactly matching, (e.g. Natural Question), an instance in the dataset can usually be organized in following format:

Desired JSON format for the output
{
    "query_id": "<query id>",
    "query_text": "<query text>",
    "query_image": "<query image>",
    "positive_document_ids": ["<passage id>", ...],
    "negative_document_ids": ["<passage id>", ...],
}
"""

data_path = Path("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/nobel_prize")

test_data = {
    "doc": read_json(data_path / "test" / "doc.json"),
    "query": read_json(data_path / "test" / "query.json"),
}

The file is of type: <class 'list'>
The file contains 989 items.
The file is of type: <class 'list'>
The file contains 3244 items.
The file is of type: <class 'list'>
The file contains 8060 items.
The file is of type: <class 'list'>
The file contains 165 items.


In [ ]:
"""
1. Relevancy Judged Dataset

If the relevancy of a passage is annotated, (e.g. MS MARCO passage ranking), an instance in the dataset can usually be organized in following format:

Desired JSON format for the output
{
   "query_id": "<query id>",
   "query": "<query text>",
   "answers": ["<answer>"],
   "positive_passages": [
     {"docid": "<passage id>",
     "title": "<passage title>",
     "text": "<passage body>"}
   ],
   "negative_passages": [
     {"docid": "<passage id>",
     "title": "<passage title>",
     "text": "<passage body>"}
   ]
}
"""

train_data = {
    "train": read_json(
        data_path / "train" / "contriever_finetune_train_v3.jsonl", jsonl=True
    ),
    "eval": read_json(
        data_path / "train" / "contriever_finetune_eval_v3.jsonl", jsonl=True
    ),
}

pprint(train_data["train"][0])

{'negative_ctxs': [{'text': 'Hélder Barbosa played for which team from 2002 to '
                            '2009?'},
                   {'text': 'Hélder Barbosa played for which team from 2006 to '
                            '2009?'}],
 'positive_ctxs': [{'text': 'Hélder Barbosa played for which team from 2010 to '
                            '2013?'},
                   {'text': 'Hélder Barbosa played for which team between Jun '
                            '2012 and Oct 2012?'}],
 'question': 'On 2 July 2010 , after helping Setúbal avoid top-flight '
             'relegation , Barbosa was released by Porto , signing a '
             'three-year contract with S.C . Braga . Rarely used in the first '
             'months , he began gaining more playing time after the January '
             '2011 departure of Matheus , who left for a team in Ukraine , and '
             'contributed four league goals in an eventual fourth-place finish '
             '.'}


In [111]:
train = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time-sensitive-qa/annotated/annotated_train.json",
    jsonl=False,
)
dev = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time-sensitive-qa/annotated/annotated_dev.json",
    jsonl=False,
)
test = read_json(
    "/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time-sensitive-qa/annotated/annotated_test.json",
    jsonl=False,
)

The file is of type: <class 'list'>
The file contains 3561 items.
The file is of type: <class 'list'>
The file contains 750 items.
The file is of type: <class 'list'>
The file contains 750 items.


In [ ]:
train_corpus = {" ".join(v["paras"]) for v in train}
dev_corpus = {" ".join(v["paras"]) for v in dev}
test_corpus = {" ".join(v["paras"]) for v in test}
final_corpus = train_corpus.union(dev_corpus).union(test_corpus)
print(len(final_corpus))

4931


In [91]:
for v in test:
    if v["link"] == "/wiki/Seth_Nana_Twumasi":
        print(v["index"])
        print(v["link"])

/wiki/Seth_Nana_Twumasi#P54
/wiki/Seth_Nana_Twumasi


In [93]:
print("Length of test corpus:", len(test))
links = [x["link"] for x in test]
print("Length of links:", len(links))
unique_links, counts = np.unique(links, return_counts=True)
indices = np.where(counts > 1)
print("Duplicate links found:", unique_links[indices])

Length of test corpus: 750
Length of links: 750
Duplicate links found: ['/wiki/Kizzmekia_Corbett']


In [83]:
unique_links[420]

'/wiki/Kizzmekia_Corbett'

In [76]:
links[indices[0]]

'/wiki/Seth_Nana_Twumasi'

## Run TSContriever evaluation script

In [ ]:
%cd /home/thuy0050/code/TS-Retriever/evaluation

from experiment import experiment

embed_model_query = "Tscontriever"
embed_model_doc = "Tscontriever"
query_embed_save_dir = "/home/thuy0050/mg61_scratch2/thuy0050/exp/ts-retriever/models/temp_embed_files/query"
doc_embed_save_dir = "/home/thuy0050/mg61_scratch2/thuy0050/exp/ts-retriever/models/temp_embed_files/docs"

experiment(embed_model_query, embed_model_doc, query_embed_save_dir, str(Path(doc_embed_save_dir) / f"{embed_model_doc}_embed_doc"))

Sun Jun  1 18:03:32 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100 80GB PCIe          On  | 00000000:65:00.0 Off |                    0 |
| N/A   33C    P0              46W / 300W |      0MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--